# Exploring before you model

Data Scientist track, lesson DS1. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** profile a dirty extract (grain, keys, gaps, copies, impossible values, two clocks and a renamed value), write the problem list with a decision for each problem, and split before any decision that uses the target.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.model_selection import train_test_split

SEED = 2026       # the seed of the whole Data Scientist track
KEY = ["user_id", "event_ts", "event_name", "platform"]      # what makes two event rows the same event

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")
con.execute(f"CREATE TABLE subscriptions AS FROM read_csv('{fetch('subscriptions')}', types={{'started_on': 'DATE', 'ended_on': 'DATE', 'updated_at': 'TIMESTAMP'}})")
users = con.sql("SELECT * FROM users").df()
events = con.sql("SELECT * FROM events").df()
subs = con.sql("SELECT * FROM subscriptions").df()

# Known answer from the dataset README: 21,841 people, 177,379 events and 3,840 subscription rows.
assert (len(users), len(events), len(subs)) == (21841, 177379, 3840)
print(f"Data loaded: {len(users):,} users, {len(events):,} events, {len(subs):,} subscription rows.")

## The modeling table

Unit 5 and Exercise 3 use the modeling table of the track, one row for each person. Lesson DS3 builds it with this query; here it is run as it is.

In [ ]:
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"{len(table):,} people in the table.")

## Unit 1: Grain, keys and gaps

**Predict first.** How many empty cells are there in `users` and `events` together? How many events belong to a person with no row in `users`, and how many users have no event at all?

In [ ]:
print(users.shape, events.shape)
print(bool(users["user_id"].is_unique), bool(events["event_id"].is_unique))
print(int(users.isna().sum().sum()), int(events.isna().sum().sum()))
no_user = ~events["user_id"].isin(users["user_id"])
no_event = ~users["user_id"].isin(events["user_id"])
orphan_events, orphan_people, idle_users = int(no_user.sum()), events.loc[no_user, "user_id"].nunique(), int(no_event.sum())
print(orphan_events, orphan_people, idle_users)
print(users.loc[no_event, "signup_date"].min().date(), users.loc[no_event, "signup_date"].max().date())

## Unit 2: Copies and impossible values

**Predict first.** `events.duplicated()` with no arguments, and `events.duplicated(subset=KEY)`: how many rows does each mark?

In [ ]:
print(int(events.duplicated().sum()), int(events.duplicated(subset=KEY).sum()))
pair = events[events.duplicated(subset=KEY, keep=False)].sort_values(KEY + ["event_id"]).head(2)
print(pair[["event_id", "event_ts", "event_name", "received_at"]].to_string(index=False))
copies = int(events.duplicated(subset=KEY).sum())
print(sorted(set(events["event_name"]) - set(events.loc[events.duplicated(subset=KEY), "event_name"])))
future = int((events["event_ts"] > events["received_at"]).sum())
print(future)

## Unit 3: Two clocks

**Predict first.** On March 17 the collector was down for part of the afternoon. For the hours 14 to 19, what will the row "arrived" show, and what will it show for hour 20?

In [ ]:
day = pd.Timestamp("2026-03-17")
by_hour = pd.DataFrame({
    "happened": events[events["event_ts"].dt.normalize() == day].groupby(events["event_ts"].dt.hour).size(),
    "arrived": events[events["received_at"].dt.normalize() == day].groupby(events["received_at"].dt.hour).size(),
}).fillna(0).astype(int)
print(by_hour.loc[13:21].T.to_string())
wait = events["received_at"] - events["event_ts"]
late = int((wait >= pd.Timedelta(hours=1)).sum())
print(late, int((wait >= pd.Timedelta(days=1)).sum()))

## Unit 4: A name that changed

**Predict first.** From May 18 the iOS app sends `paywall_viewed` for the paywall. Which platforms still send `view_paywall` from May 18, and does any platform send both names?

In [ ]:
paywall = events[events["event_name"].isin(["view_paywall", "paywall_viewed"])]
from_may_18 = paywall["event_ts"] >= pd.Timestamp("2026-05-18")
print(pd.crosstab([from_may_18.rename("from May 18"), paywall["platform"]], paywall["event_name"]).to_string())
renamed = int((events["event_name"] == "paywall_viewed").sum())
print(renamed)

## Unit 5: The problem list, and the split

**Predict first.** Seven problems stand in the list. Before you run the cell, write fix, flag or leave beside each one: events of people with no user row, users with no event, retry copies, events dated after they arrived, events that arrived late, the renamed event, and signups with under 30 days of follow-up.

In [ ]:
incomplete = int((users["signup_date"] >= pd.Timestamp("2026-06-01")).sum())     # signed up from June 1 on: fewer than 30 days of follow-up before the export
problems = pd.DataFrame([
    ("events of people with no user row", orphan_events, "flag"),
    ("users with no event", idle_users, "leave"),
    ("retry copies of events", copies, "fix"),
    ("events dated after they arrived", future, "flag"),
    ("events that arrived an hour or more late", late, "fix"),
    ("paywall_viewed, the renamed event", renamed, "fix"),
    ("signups with under 30 days of follow-up", incomplete, "fix"),
], columns=["problem", "rows", "decision"])
print(problems.to_string(index=False))
train, test = train_test_split(table, test_size=0.25, random_state=SEED, stratify=table["churned"])      # the split of the track, made before the target is used
print(len(train), len(test))
print(train.groupby("region")["churned"].mean().round(3).to_dict())
gone = events[~events["user_id"].isin(users["user_id"])]                  # the 40 deleted accounts left events but no user row
start = gone[gone["event_name"] == "signup_complete"].set_index("user_id")["event_ts"]
act = gone[gone["event_name"].isin(["session_start", "add_expense"])]
days = (act["event_ts"] - act["user_id"].map(start)).dt.total_seconds() / 86400
churned_gone = len(start) - act.loc[(days > 7) & (days <= 30), "user_id"].nunique()     # same label: no visit and no expense in days 8 to 30
january = train.loc[train["signup_date"] < pd.Timestamp("2026-02-01"), "churned"]      # the target, on the training part only
print(len(start), churned_gone, round(100 * january.mean(), 2), round(100 * (january.sum() + churned_gone) / (len(january) + len(start)), 2))

## Finish it

### Exercise 1: Finish it

**Task:** **Which events have copies?** The starter marks the extra copies of events, one row for each copy after the first. Put in `answer_1` the number of extra copies for each event name, as a dictionary `{name: count}`.

**Expected:** `answer_1` is a dictionary with 6 event names as keys and whole numbers as values.

In [ ]:
extra = events[events.duplicated(subset=KEY)]          # given: the extra copies, one row for each copy after the first
answer_1 = None  # replace None with the number of extra copies for each event name, as a dictionary {name: count}

In [ ]:
_ = check("Exercise 1", "answer_1", {'add_expense': 406, 'session_start': 202, 'plus_feature_used': 133, 'view_paywall': 64, 'start_checkout': 21, 'paywall_viewed': 9}, traps=[({'add_expense': 811, 'session_start': 404, 'plus_feature_used': 266, 'view_paywall': 128, 'start_checkout': 42, 'paywall_viewed': 18}, "That marks both rows of every pair (keep=False). Count only the extra copies, the rows after the first.")])

<details>
<summary>Hint</summary>

Count the rows of `extra` for each event name: `value_counts()` on one column does it, and `.items()` gives the pairs.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = {name: int(n) for name, n in extra["..."].value_counts().items()}
```

</details>

<details>
<summary>Solution</summary>

```python
answer_1 = {name: int(n) for name, n in extra["event_name"].value_counts().items()}
```

The extra copies are add_expense 406, session_start 202, plus_feature_used 133, view_paywall 64, start_checkout 21, paywall_viewed 9, 835 in all. Marking every row of every pair (`keep=False`) would count 1,669, both rows of each pair; the extra copies are the rows to remove.

</details>

## On your own

### Exercise 2: On your own

**Task:** **How dirty is the subscriptions table?** Before you start, say in one sentence what the result looks like: two whole numbers, the second much smaller than 3,840. `subs` holds the 3,840 rows of `subscriptions`: one row for each saved version of a subscription. Put the list [rows that are exact copies of an earlier row, subscriptions that have more than one row] in `answer_2`.

**Expected:** `answer_2` is a list of two whole numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [97, 874], traps=[([935, 874], "The first number counts rows that repeat an earlier row in every column: call duplicated() with no arguments.")])

<details>
<summary>Hint</summary>

`subs.duplicated()` with no arguments marks rows that repeat an earlier row in every column. For the second number, count the rows of each `subscription_id` with `groupby(...).size()` and count the ids above 1.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Two whole numbers: about a hundred exact copies, and several hundred subscriptions with more than one row."*

```python
answer_2 = [int(subs.duplicated().sum()), int((subs.groupby(...).size() > 1).sum())]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = [int(subs.duplicated().sum()), int((subs.groupby("subscription_id").size() > 1).sum())]
```

97 rows repeat an earlier row in every column, and 874 of the 2,905 subscriptions have more than one row. Most of those extra rows are not copies: a subscription is saved again when it changes, such as a cancel, so its latest row (by `updated_at`) is the truth. Counting `subscription_id` as a key that must be unique would flag 874 false problems.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague's problem list says: "fix: drop the people with no activity in week 1; zero looks like broken tracking." The starter drops them and checks that the churn rate of the rest hardly moves. Change the last line to report what the drop deletes: the share of all churners in the training part who had no activity in week 1, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0 and 1.

In [ ]:
kept = train[train["active_days_7d"] > 0]       # colleague: "no activity in week 1 is broken tracking, so I dropped those rows. The churn rate hardly moves"
answer_3 = round(float(kept["churned"].mean()), 3)      # colleague's check: the churn rate after the drop
# change the last line to report what the drop deletes: the share of all churners in the training part who had no activity in week 1

In [ ]:
_ = check("Exercise 3", "answer_3", 0.323, traps={0.491: "That is the colleague's number, the churn rate after the drop. Report the share of all churners that the drop deletes."}, tol=5e-4)

<details>
<summary>Hint</summary>

The churners in `kept` are the churners that survive the drop. What share of the training part's churners is that, and what is left over?

</details>

<details>
<summary>Half-solution</summary>

```python
answer_3 = round(float(1 - kept["churned"].sum() / train["..."].sum()), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = round(float(1 - kept["churned"].sum() / train["churned"].sum()), 3)
```

The drop deletes 32.3 percent of the churners, while it removes 25.0 percent of the people. The churn rate of the rest, 0.491 against 0.544 before, looks like a small move and hides this. The people with no activity are not missing data: zero is a value, not a gap (unit 1 found no empty cell), and 70.4 percent of them churned. They are the very rows the model must learn from.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")